# Four-language reasoning on Gemma 4 — Kaggle

Swahili · Wolof · English · French, keeping the model's **thinking**, not just
its answers.

Run the cells top to bottom. **Cell 2 may restart the session once** — that is
expected; when it does, just run it again and carry on.

## Before you start

1. **Settings → Accelerator → **GPU T4 x2** (or P100)**
2. **Settings → **Internet: On** — the model download needs it**
3. Accept the Gemma 4 licence on its Hugging Face model page.
4. Put your Hugging Face token in ****Add-ons → Secrets**** as `HF_TOKEN`.

## What this does, in order

| cell | step |
|---|---|
| 2 | environment: GPU, dependencies, restart if needed |
| 3 | fresh clone, authenticate, run the test suite |
| 4 | confirm the thinking format, build the data |
| 5 | smoke test — 4 items, ~2 minutes |
| 6 | **baseline** — 48 items, the "before" measurement |
| 8 | QLoRA fine-tune |
| 9 | evaluate and compare against the baseline |
| 10 | export and save |

The baseline comes before the fine-tune on purpose. Without a "before" there is
nothing to prove an improvement against, and a good-looking number after
training proves nothing on its own.

> **Data status.** Nothing in the training sample has been verified by a native
> speaker; Wolof is unreviewed machine translation. Cell 8 passes
> `--allow-unverified`, which stamps the model card as a pipeline test. Numbers
> from such a run must not be reported. The baseline in cell 6 is unaffected —
> it uses the untouched base model and is a real measurement.


## 2. Environment

Installs dependencies and checks the GPU. **This cell may restart the session** — if it does, run it again.

In [ ]:
# CELL 2 — environment. May restart the session once; that is expected.
import subprocess, sys, os

PACKAGES = ["transformers>=5.16", "peft>=0.14", "accelerate>=1.0",
            "bitsandbytes>=0.44", "datasets>=3.0"]

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "NO GPU — enable it in Settings > Accelerator")

print("\ninstalling (quiet, ~1 min) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES],
               check=True)

def _version(text):
    out = []
    for part in text.split(".")[:3]:
        digits = "".join(c for c in part if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)

import transformers
print("transformers", transformers.__version__)

# transformers below 5.16 cannot build a gemma4 config, and this process is
# still running whatever was imported before the upgrade -- so restart.
if _version(transformers.__version__) < (5, 16):
    print("\n" + "=" * 70)
    print("RESTARTING THE SESSION to pick up the upgrade.")
    print("This is normal. When it finishes, RUN THIS CELL AGAIN and continue.")
    print("=" * 70)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    import torch
    BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    print(f"bfloat16 supported: {BF16}  ->  precision flag "
          f"{'(bf16, default)' if BF16 else '--fp16'}")
    print("\nenvironment ready — continue to cell 3")


## 3. Code and credentials

Clones fresh every run, then runs the test suite. If the tests fail, stop: every later number would be meaningless.

In [ ]:
# CELL 3 — fresh clone, authentication, self-test.
import os, shutil, subprocess, sys

ROOT    = "/kaggle/working"
PROJECT = "/kaggle/working/repo/mlr"
BRANCH  = "claude/gemma-4-multilingual-reasoning-xqse1j"
REPO    = "https://github.com/Joe254h/python.git"

# Deleted and re-cloned every run. A stale checkout is invisible and produces
# "no such file" errors for files that plainly exist on the branch.
shutil.rmtree(f"{ROOT}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "--single-branch", REPO, f"{ROOT}/repo"], check=True)
os.chdir(PROJECT)
print("project:", PROJECT)
print("commit :", subprocess.run(["git", "log", "--oneline", "-1"],
                                 capture_output=True, text=True).stdout.strip())

# Hugging Face token. Gemma 4 is gated.
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as exc:
    print(f"NO HF TOKEN ({exc}) — add it under Add-ons > Secrets, "
          f"and accept the Gemma 4 licence on its model page.")

# The harness must be trustworthy before any number it produces means anything.
print("\nrunning the test suite ...")
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                        cwd=PROJECT, capture_output=True, text=True)
print(result.stderr.strip().splitlines()[-1])
assert result.returncode == 0, "tests failed — stop here, later numbers are meaningless"


## 4. Thinking format, then data

Confirms how this model marks its reasoning block, then builds the 48-item held-out set and the 20-example sample.

In [ ]:
# CELL 4 — confirm the thinking format, then build the data.
import os, subprocess, sys, torch

PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)

BASE_MODEL = os.environ.get("BASE_MODEL", "google/gemma-4-E4B-it")
FP16 = "" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "--fp16"
os.environ["BASE_MODEL"], os.environ["FP16"] = BASE_MODEL, FP16
print(f"model {BASE_MODEL}   precision flag {FP16 or '(bf16)'}\n")

# Tokenizer only, no weights, seconds. Confirms the delimiters and -- more
# importantly -- whether the template opens the reasoning block itself.
subprocess.run([sys.executable, "scripts/inspect_chat_template.py",
                "--model", BASE_MODEL], cwd=PROJECT, check=False)

print("\n" + "=" * 70 + "\nbuilding data\n" + "=" * 70)
for script in ("scripts/build_eval_set.py", "scripts/build_sample.py"):
    subprocess.run([sys.executable, script], cwd=PROJECT, check=True)


## 5. Smoke test

Four items, one per language.

In [ ]:
# CELL 5 — smoke test: 4 items, one per language. ~2 minutes.
# Catches a broken path here instead of an hour into the full run.
import os, torch
PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)
BASE_MODEL = os.environ.setdefault("BASE_MODEL", "google/gemma-4-E4B-it")
# Turing (T4) and Pascal (P100) have no bfloat16. Detected, never asked.
FP16 = "" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "--fp16"
os.environ["FP16"] = FP16

!cd /kaggle/working/repo/mlr && python scripts/run_eval.py --mode baseline \
    --base $BASE_MODEL --out results_smoke --limit 1 $FP16

print("\nCheck three things above before continuing:")
print("  1. 'architecture: gemma4' appeared before any download")
print("  2. format is 100% — the think block parsed")
print("  3. the reasoning is actually in the requested language")


## 6. Baseline — the measurement everything else is judged against

In [ ]:
# CELL 6 — THE BASELINE. 48 items, untouched base model. 20-60 minutes.
# This is the "before" the whole project is measured against.
import os
PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)

!cd /kaggle/working/repo/mlr && python scripts/run_eval.py --mode baseline \
    --base $BASE_MODEL --out results $FP16


In [ ]:
# CELL 7 — the baseline table. Keep this; it is what the fine-tune must beat.
PROJECT = "/kaggle/working/repo/mlr"
print(open(f"{PROJECT}/results/baseline/baseline_table.txt").read())


## 8. Fine-tune with QLoRA

Base frozen in 4-bit, adapters trained on top. LoRA targets are discovered from
the loaded model rather than hardcoded, so this still works if Gemma 4 names its
projections differently from earlier releases.

`--allow-unverified` is here because the Wolof native review has not happened
yet. It stamps the model card as a pipeline test. **Remove it once the review is
signed off**, and the resulting numbers become reportable.

In [ ]:
# CELL 8 — QLoRA fine-tune. 15-40 minutes for 80 rows.
import os
PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)

!cd /kaggle/working/repo/mlr && python scripts/train_lora.py \
    --data data/sample20/sample20.jsonl \
    --base $BASE_MODEL --out artifacts/adapter \
    --epochs 3 --lora-r 16 --allow-unverified $FP16


## 9. Evaluate and compare

In [ ]:
# CELL 9 — evaluate the fine-tune on the SAME held-out set, and compare.
import os, torch
PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)
BASE_MODEL = os.environ.setdefault("BASE_MODEL", "google/gemma-4-E4B-it")
# Turing (T4) and Pascal (P100) have no bfloat16. Detected, never asked.
FP16 = "" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "--fp16"
os.environ["FP16"] = FP16
import json

!cd /kaggle/working/repo/mlr && python scripts/run_eval.py --mode adapter \
    --base $BASE_MODEL --adapter artifacts/adapter --out results $FP16

base  = json.load(open(f"{PROJECT}/results/baseline/baseline_report.json"))["summary"]
tuned = json.load(open(f"{PROJECT}/results/finetuned/baseline_report.json"))["summary"]
card  = json.load(open(f"{PROJECT}/artifacts/adapter/model_card.json"))

METRICS = ("reasoning_correct", "reasoning_lang_ok", "answer_lang_ok",
           "collapse_to_english", "correct_and_in_language")
deltas = {lang: {m: round(tuned["by_language"][lang][m] - base["by_language"][lang][m], 4)
                 for m in METRICS}
          for lang in base["by_language"]}

json.dump({"base_model": os.environ["BASE_MODEL"], "adapter": "artifacts/adapter",
           "model_card": card, "baseline": base, "finetuned": tuned, "deltas": deltas},
          open(f"{PROJECT}/results/comparison.json", "w"), indent=2, ensure_ascii=False)

print(f"\n{'lang':>5}  {'correct':>9} {'in-language':>12} {'collapse-EN':>12}")
for lang, d in deltas.items():
    print(f"{lang:>5}  {d['reasoning_correct']:>+9.0%} "
          f"{d['correct_and_in_language']:>+12.0%} {d['collapse_to_english']:>+12.0%}")
print("\nFor collapse-to-English, negative is the improvement.")


## 10. Export and save

In [ ]:
# CELL 10 — export for the web app, and persist everything.
import os, torch
PROJECT = "/kaggle/working/repo/mlr"
os.chdir(PROJECT)
BASE_MODEL = os.environ.setdefault("BASE_MODEL", "google/gemma-4-E4B-it")
# Turing (T4) and Pascal (P100) have no bfloat16. Detected, never asked.
FP16 = "" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "--fp16"
os.environ["FP16"] = FP16
import shutil, pathlib, subprocess, sys

subprocess.run([sys.executable, "scripts/export_model.py",
                "--adapter", "artifacts/adapter", "--base", os.environ["BASE_MODEL"],
                "--out", "artifacts/serve", "--kind", "adapter"], cwd=PROJECT, check=True)

dest = pathlib.Path("/kaggle/working/gemma4-mlr")
dest.mkdir(parents=True, exist_ok=True)
for name in ("artifacts/serve", "results"):
    shutil.copytree(f"{PROJECT}/{name}", dest / pathlib.Path(name).name,
                    dirs_exist_ok=True)

print(f"saved to {dest}\n")
for p in sorted(dest.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(dest)}  ({p.stat().st_size/1e6:.1f} MB)")
print("\nEverything under /kaggle/working is kept when the notebook commits, and can be attached to another notebook or downloaded.")
